# 02e | Data Cleaning

Real take-home and OA (online assessment) data is always messy. This lesson covers four things: missing values, type conversion, dates, and strings.

**After this lesson you can:** turn a table with missing values, dirty entries, and dates stored as strings into one that's ready for analysis. About 50 minutes.

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()

raw = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "segment": [" A", "a", "B", "b ", "B", "A"],
    "date": ["2026-01-01", "2026-01-03", "2026-01-02", "2026-01-04", "bad", "2026-01-05"],
    "revenue": ["10", "20", None, "50", "40", "n/a"],
    "active": [1, 1, 0, 1, 1, 1],
})
raw

## 1. Missing values

In pandas, missing values show up as `NaN` (numbers), `None`, or `NaT` (dates); they're the equivalent of R's `NA`.

| R | pandas |
|---|---|
| `is.na(x)` | `s.isna()` |
| `colSums(is.na(df))` | `df.isna().sum()` |
| `na.omit(df)` | `df.dropna()` |
| `tidyr::drop_na(df, col)` | `df.dropna(subset=["col"])` |
| `replace_na()` | `s.fillna(value)` |

⚠️ **The default is the opposite of R:** pandas' `mean()`, `sum()`, etc. **skip missing values by default** (as if R's `na.rm = TRUE` were always on). So you won't get NA as a result, but you may not notice that values are missing. Make a habit of checking `df.isna().sum()` first.

In [ ]:
print(users.isna().sum())
s = pd.Series([1.0, np.nan, 3.0])
print(s.mean())             # 2.0: the missing value is skipped
print(s.fillna(0).mean())   # 1.33: filling with 0 gives a different result

print(len(users), len(users.dropna()), len(users.dropna(subset=["country"])))
# dropna() drops any row with a missing value in any column; subset= looks only at the given columns

## 2. Type conversion

`raw["revenue"]` looks numeric but is actually strings (`dtype` is `object`), so it can't be summed.

- `pd.to_numeric(s, errors="coerce")`: converts to numbers; anything that can't be converted becomes NaN;
- `pd.to_datetime(s, errors="coerce")`: converts to dates; anything that can't be converted becomes NaT;
- `s.astype(int)` / `s.astype(str)`: use when you're sure the conversion will work.

`errors="coerce"` means "don't raise on dirty values; mark them as missing."

In [ ]:
print(raw.dtypes)
print(pd.to_numeric(raw["revenue"], errors="coerce"))
print(pd.to_datetime(raw["date"], errors="coerce"))

## 3. Strings: `.str`

To operate on a whole column of strings, put `.str` before the method: `s.str.lower()`, `s.str.strip()`, `s.str.contains("abc")`, `s.str.replace("a", "b")`, `s.str.len()`. The method names are the same as the single-string methods in 01f.

In [ ]:
print(raw["segment"].unique())
print(raw["segment"].str.strip().str.upper().unique())

### ✏️ Exercise 1 (Write it yourself)

Clean `raw` into `clean` (start with `raw.copy()`):

1. Strip leading/trailing spaces from `segment` and convert it to uppercase;
2. Convert `revenue` to numbers, setting anything unconvertible to missing;
3. Convert `date` to dates, setting anything unconvertible to missing;
4. Drop rows where `date` is missing (**keep** the missing values in `revenue`).

In [ ]:
clean = raw.copy()
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
clean = raw.copy()
clean["segment"] = clean["segment"].str.strip().str.upper()
clean["revenue"] = pd.to_numeric(clean["revenue"], errors="coerce")
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean = clean.dropna(subset=["date"])
clean
```

</details>

**🗣 Say it out loud:** Why shouldn't you just fill in missing values with the mean as soon as you see them?

<details>
<summary><b>Reference answer</b></summary>

First ask how the missingness arose: it may carry meaning itself, or it may be related to other variables or even to the missing value itself (MCAR / MAR / MNAR). Mechanically filling in the mean has two problems. First, all the imputed values pile up at the mean, which artificially shrinks the variance and weakens correlations with other variables. Second, it treats "unknown" as "known," so downstream standard errors are too small. Common alternatives: add a missingness indicator, use model-based or multiple imputation, or use a model that handles missing values directly. For prediction, the statistics used for imputation must be computed on the training set only.

**Short version:** Mean imputation shrinks the variance and weakens correlations, and it ignores why the data are missing. I'd first look at the missingness mechanism, consider a missing indicator or model-based / multiple imputation, and fit any imputation on the training set only.

</details>

## 4. Dates: `.dt`

Once a column is a datetime type, use `.dt` to extract parts such as year, month, and day (R's lubridate):

| You want | Code |
|---|---|
| Year / month / day | `s.dt.year` / `s.dt.month` / `s.dt.day` |
| Day of the week | `s.dt.day_name()` |
| Month (e.g. 2026-01) | `s.dt.to_period("M")` or `s.dt.strftime("%Y-%m")` |
| Days between two dates | `(s2 - s1).dt.days` |
| Add 7 days | `s + pd.Timedelta(days=7)` |

In [ ]:
o = orders.copy()
o["order_date"] = pd.to_datetime(o["order_date"])
o["month"] = o["order_date"].dt.strftime("%Y-%m")
o["weekday"] = o["order_date"].dt.day_name()
o["days_since_jan1"] = (o["order_date"] - pd.Timestamp("2026-01-01")).dt.days
o[["order_id", "order_date", "month", "weekday", "days_since_jan1"]].head(4)

### ✏️ Exercise 2 (Write it yourself)

Compute the total amount of completed orders per month. The result has two columns: `month` (a string like `"2026-01"`) and `total`; store it in `out`.

In [ ]:
o = orders.copy()
# Write your code below

<details>
<summary><b>Hint</b></summary>

Convert to dates first, then use `.dt.strftime("%Y-%m")` to get a month column, then filter, group, and sum.

</details>

<details>
<summary><b>Answer</b></summary>

```python
o = orders.copy()
o["order_date"] = pd.to_datetime(o["order_date"])
o["month"] = o["order_date"].dt.strftime("%Y-%m")
out = (
    o[o["status"] == "completed"]
    .groupby("month")["amount"].sum()
    .reset_index()
    .rename(columns={"amount": "total"})
)
out
```

</details>

### ✏️ Exercise 3 (Write it yourself)

For each user who has placed an order, compute **how many days passed between signing up and their first order**. The result has two columns, `user_id` and `days`; store it in `out`.

Steps: each user's earliest order date → merge with `users` to get `signup_date` → subtract the two dates.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

You can take `.min()` of the date strings directly to get the earliest (because the format is year-month-day). Before subtracting, apply `pd.to_datetime` to both columns; after subtracting, use `.dt.days`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
first = orders.groupby("user_id")["order_date"].min().reset_index()
m = first.merge(users[["user_id", "signup_date"]], on="user_id")
m["days"] = (pd.to_datetime(m["order_date"]) - pd.to_datetime(m["signup_date"])).dt.days
out = m[["user_id", "days"]]
out
```

</details>

## 5. Binning and custom transformations

- `pd.cut(s, bins=[...], labels=[...])`: bins values at the given cut points (R's `cut`);
- `pd.qcut(s, 4)`: splits into 4 groups by quantile;
- `s.map(dict)`: replaces values using a lookup table;
- `s.apply(function)`: calls a function on each element. If a whole-column operation works, don't use `apply`; it's much slower.

`lambda x: x * 2` is a "throwaway little function," equivalent to R's `function(x) x * 2`, often used inside `apply` and `sorted(key=...)`.

In [ ]:
o = orders.copy()
o["bucket"] = pd.cut(o["amount"], bins=[0, 20, 40, np.inf], labels=["<20", "20-39", ">=40"], right=False)
o["status_en"] = o["status"].map({"completed": "Completed", "refunded": "Refunded"})
o["amount_label"] = o["amount"].apply(lambda a: f"${a:.0f}")
o[["amount", "bucket", "status_en", "amount_label"]].head(5)

After binning, "how many in each bin": after `groupby`, call `.count()` on any column (counts non-missing values) or `.size()` (counts rows). As in 02c, follow it with `reset_index()` and `rename`:

In [ ]:
(o.groupby("status")["order_id"].count()
  .reset_index()
  .rename(columns={"order_id": "n"}))

### ✏️ Exercise 4 (Write it yourself)

Split all orders into three bins by amount: under 20 is `"low"`, 20 to 39 is `"mid"`, 40 and above is `"high"`. Count the orders in each bin, as two columns `bucket` and `n`, and store the result in `out`.

In [ ]:
o = orders.copy()
# Write your code below

<details>
<summary><b>Hint</b></summary>

You can nest two `np.where` calls, or use `pd.cut(..., right=False)`. Then count with `groupby("bucket")`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
o = orders.copy()
o["bucket"] = np.where(o["amount"] < 20, "low", np.where(o["amount"] < 40, "mid", "high"))
out = o.groupby("bucket")["order_id"].count().reset_index().rename(columns={"order_id": "n"})
out
```

</details>

**🗣 Say it out loud:** Why is it sloppy to "standardize the full dataset first, then split it into training and test sets"?

<details>
<summary><b>Reference answer</b></summary>

The mean and standard deviation used for standardization are computed from all the data, including information from the test set. That's data leakage: the test set is no longer "unseen data," and the evaluation will be optimistic. The correct approach is to compute these parameters on the training set only, then apply the same parameters to the test set. In practice, put the preprocessing and the model in the same Pipeline, so that during cross-validation each fold automatically fits on its training portion only.

**Short version:** Because the scaler's mean and standard deviation would be computed using test data, which leaks information and makes evaluation optimistic. Fit preprocessing on the training set only, ideally inside a pipeline so it is refit within each CV fold.

</details>

## Summary

- Check `df.isna().sum()` first; pandas statistics skip missing values by default;
- Use `pd.to_numeric` / `pd.to_datetime` with `errors="coerce"`;
- Use `.str` for whole-column string operations and `.dt` for dates;
- Bin with `pd.cut`; build conditional columns with `np.where`.

Move on to **02f** to use these tools on interview questions.